# YOLOv8n Acne Detection — Hyperparameter Tuning + WandB

**Pipeline:**
1. Install dependensi & konfigurasi WandB API key
2. Konfigurasi path & parameter dasar
3. Modifikasi arsitektur YOLOv8n (EMA Attention di neck)
4. Hyperparameter Tuning dengan Optuna (lr, weight_decay, batch, loss coefficients)
5. Training final dengan best params + WandB logging
6. Evaluasi & visualisasi (confusion matrix, PR curve, inference samples)

## Cell 1 — Install Dependensi & Input WandB API Key
Jalankan sekali saja. Setelah install selesai, isi `WANDB_API_KEY` dengan key Anda.

In [1]:
# ─── Install dependensi (jalankan sekali) ───────────────────────────────────
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])
pip_install('ultralytics>=8.2.0')
pip_install('wandb>=0.17.0')
pip_install('optuna>=3.6.0')
pip_install('matplotlib', 'numpy', 'pillow', 'pyyaml')
print('✓ Semua dependensi berhasil diinstall')
# ─── Konfigurasi WandB API Key ──────────────────────────────────────────────
import wandb
# ⚠️  GANTI dengan API key WandB Anda (https://wandb.ai/settings)
WANDB_API_KEY = "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"
import os
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
wandb.login(key=WANDB_API_KEY, relogin=True)
print('✓ WandB login berhasil')

✓ Semua dependensi berhasil diinstall


wandb: ERROR Failed to detect the name of this notebook. You can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/rof/.netrc
wandb: Currently logged in as: delimabrpurba72 (delimabrpurba72-universitas-prasetiya-mulya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB login berhasil


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\delim\_netrc
wandb: Currently logged in as: delimabrpurba72 (delimabrpurba72-universitas-prasetiya-mulya) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✓ WandB login berhasil


## Cell 2 — Konfigurasi Path & Parameter Dasar

In [2]:
from pathlib import Path
import yaml, os, sys, torch

BASE_DIR   = Path(r"/home/rof/Delima/Delima/DATA FIX UNTUK TRAIN")
DATA_YAML  = BASE_DIR / 'data.yaml'
OUTPUT_DIR = BASE_DIR / 'runs'
OUTPUT_DIR.mkdir(exist_ok=True)

# Buat data_runtime.yaml dengan path absolut agar cross-platform
with open(DATA_YAML) as f:
    _raw_cfg = yaml.safe_load(f)

_runtime_cfg = _raw_cfg.copy()
for _key in ('train', 'val', 'test'):
    if _key in _runtime_cfg:
        _p = Path(_runtime_cfg[_key])
        if not _p.is_absolute():
            _runtime_cfg[_key] = str(BASE_DIR / _p)

DATA_YAML_RUNTIME = OUTPUT_DIR / 'data_runtime.yaml'
with open(DATA_YAML_RUNTIME, 'w') as f:
    yaml.dump(_runtime_cfg, f, default_flow_style=False, allow_unicode=True)

# Gunakan runtime yaml untuk training
DATA_YAML = DATA_YAML_RUNTIME

cfg = _runtime_cfg
CLASS_NAMES = cfg['names']
NUM_CLASSES = cfg['nc']
print(f'BASE_DIR : {BASE_DIR}')
print(f'Dataset  : {DATA_YAML}')
print(f'Kelas    : {NUM_CLASSES} → {CLASS_NAMES}')
print(f'Train dir: {cfg.get("train", "N/A")}')
print(f'Val dir  : {cfg.get("val", "N/A")}')
print(f'GPU      : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (tidak ada GPU)"}')

# ─── Nama proyek & eksperimen di WandB ──────────────────────────────────────
WANDB_PROJECT = "acne-detection-yolov8n"
WANDB_ENTITY  = None    # Ganti dengan username WandB Anda jika perlu, atau biarkan None

# ─── Parameter default (akan di-override setelah tuning) ────────────────────
DEFAULT_PARAMS = dict(
    epochs       = 200,
    imgsz        = 640,
    device       = 0 if torch.cuda.is_available() else 'cpu',
    workers      = 4,
    patience     = 20,       # early stopping
    # Hyperparams yang akan di-tune:
    lr0          = 0.01,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    batch        = 16,
    box          = 7.5,
    cls          = 0.5,
    dfl          = 1.5,
)
print('\n✓ Konfigurasi siap')

BASE_DIR : /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN
Dataset  : /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/runs/data_runtime.yaml
Kelas    : 9 → ['atrophic_scar', 'comedo', 'hypertrophic_scar', 'melasma', 'nevus', 'nodule', 'other', 'papule', 'pustule']
Train dir: /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/train/images
Val dir  : /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/test/images
GPU      : NVIDIA GeForce RTX 3060

✓ Konfigurasi siap


## Cell 3 — Modifikasi Arsitektur YOLOv8n

Strategi:
1. **EMA (Efficient Multi-Scale Attention)** — disematkan di neck (setelah setiap C2f pada neck)
2. **PIoU Loss** — pengganti CIoU untuk regresi bounding box lebih presisi pada objek kecil

Implementasi dilakukan via *monkey-patch* pada modul `ultralytics` tanpa mengubah file instalasi.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ═══════════════════════════════════════════════════════════════════════════════
# 1.  EFFICIENT MULTI-SCALE ATTENTION (EMA)
#     Referensi: "Efficient Multi-Scale Attention Module with Cross-Spatial
#     Learning" (2023).  Implementasi ringan berbasis grouped depthwise conv.
# ═══════════════════════════════════════════════════════════════════════════════

class EMA(nn.Module):
    """Efficient Multi-Scale Attention — plug-in attention module untuk neck YOLO."""

    def __init__(self, channels: int, factor: int = 8):
        super().__init__()
        self.groups = factor
        assert channels // self.groups > 0, "channels harus kelipatan factor"
        self.softmax  = nn.Softmax(dim=-1)
        self.agp      = nn.AdaptiveAvgPool2d((1, 1))
        self.pool_h   = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w   = nn.AdaptiveAvgPool2d((1, None))
        self.gn       = nn.GroupNorm(channels // self.groups, channels // self.groups)
        self.conv1x1  = nn.Conv2d(channels // self.groups, channels // self.groups,
                                   kernel_size=1, bias=False)
        self.conv3x3  = nn.Conv2d(channels // self.groups, channels // self.groups,
                                   kernel_size=3, padding=1, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c, h, w = x.size()
        # Bagi channel menjadi G grup
        group_x = x.reshape(b * self.groups, -1, h, w)  # (B*G, C/G, H, W)
        # Cross-spatial: pool horizontal & vertikal
        x_h = self.pool_h(group_x)                       # (B*G, C/G, H, 1)
        x_w = self.pool_w(group_x).permute(0, 1, 3, 2)   # (B*G, C/G, W, 1) → transpose
        hw  = self.conv1x1(torch.cat([x_h, x_w], dim=2)) # (B*G, C/G, H+W, 1)
        x_h, x_w = torch.split(hw, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)
        # Attention map
        a1  = torch.sigmoid(self.gn(x_h))
        a2  = torch.sigmoid(self.gn(x_w))
        group_x = group_x * a1.expand_as(group_x) * a2.expand_as(group_x)
        # Channel-wise re-calibration
        x1  = self.agp(group_x)    # (B*G, C/G, 1, 1)
        x2  = self.conv3x3(group_x).reshape(b * self.groups, -1, 1)  # (B*G, C/G*H*W, 1)
        # dot-product attention across spatial
        x11 = self.softmax(x1.reshape(b * self.groups, -1, 1))        # (B*G, C/G, 1)
        x12 = x1.reshape(b * self.groups, -1, 1)                      # (B*G, C/G, 1)
        x21 = self.softmax(x2)                                         # (B*G, C/G*H*W, 1)
        x22 = x2                                                        # (B*G, C/G*H*W, 1)
        weights = (torch.matmul(x11.transpose(1,2), x12) + 
                   torch.matmul(x21.transpose(1,2), x22)).reshape(b * self.groups, 1, h, w)
        out = group_x * weights.sigmoid().expand_as(group_x)
        return out.reshape(b, c, h, w)


# ═══════════════════════════════════════════════════════════════════════════════
# 2.  POWERFUL IoU (PIoU) LOSS
#     Referensi: "PIoU Loss: Towards Accurate Oriented Object Detection
#     in Complex Environments" — adaptasi untuk axis-aligned bounding boxes.
#     PIoU menambahkan penalti berbasis jarak Euclidean antar sudut kotak
#     untuk mencegah box enlargement pada objek kecil.
# ═══════════════════════════════════════════════════════════════════════════════

def piou_loss(pred: torch.Tensor, target: torch.Tensor,
              alpha: float = 1.0, beta: float = 1.0) -> torch.Tensor:
    """
    PIoU loss untuk axis-aligned bounding boxes.
    pred, target: (N, 4) format (x1, y1, x2, y2)
    Returns: scalar loss
    """
    # --- IoU base ---
    inter_x1 = torch.max(pred[:, 0], target[:, 0])
    inter_y1 = torch.max(pred[:, 1], target[:, 1])
    inter_x2 = torch.min(pred[:, 2], target[:, 2])
    inter_y2 = torch.min(pred[:, 3], target[:, 3])
    inter_w  = (inter_x2 - inter_x1).clamp(min=0)
    inter_h  = (inter_y2 - inter_y1).clamp(min=0)
    inter    = inter_w * inter_h
    area_p   = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
    area_t   = (target[:, 2] - target[:, 0]) * (target[:, 3] - target[:, 1])
    union    = area_p + area_t - inter + 1e-7
    iou      = inter / union

    # --- Enclosing box diagonal (CIoU component) ---
    enc_x1   = torch.min(pred[:, 0], target[:, 0])
    enc_y1   = torch.min(pred[:, 1], target[:, 1])
    enc_x2   = torch.max(pred[:, 2], target[:, 2])
    enc_y2   = torch.max(pred[:, 3], target[:, 3])
    c2       = (enc_x2 - enc_x1) ** 2 + (enc_y2 - enc_y1) ** 2 + 1e-7

    # --- Center distance squared ---
    p_cx = (pred[:, 0] + pred[:, 2]) / 2
    p_cy = (pred[:, 1] + pred[:, 3]) / 2
    t_cx = (target[:, 0] + target[:, 2]) / 2
    t_cy = (target[:, 1] + target[:, 3]) / 2
    rho2 = (p_cx - t_cx) ** 2 + (p_cy - t_cy) ** 2

    # --- PIoU corner penalty (Euclidean distance antara 4 sudut) ---
    # Sudut prediksi & target
    corner_dist = (
        (pred[:, 0] - target[:, 0]) ** 2 + (pred[:, 1] - target[:, 1]) ** 2 +  # TL
        (pred[:, 2] - target[:, 2]) ** 2 + (pred[:, 1] - target[:, 1]) ** 2 +  # TR
        (pred[:, 0] - target[:, 0]) ** 2 + (pred[:, 3] - target[:, 3]) ** 2 +  # BL
        (pred[:, 2] - target[:, 2]) ** 2 + (pred[:, 3] - target[:, 3]) ** 2    # BR
    )

    piou = iou - rho2 / c2 - alpha * corner_dist / (c2 + 1e-7) * beta
    return 1 - piou.mean()


# ═══════════════════════════════════════════════════════════════════════════════
# 3.  PATCH ultralytics — inject EMA ke neck YOLOv8n & PIoU ke loss
# ═══════════════════════════════════════════════════════════════════════════════

from ultralytics import YOLO
from ultralytics.nn.modules import C2f
import ultralytics.utils.loss as ult_loss

# --- Patch: ganti BboxLoss.forward agar menggunakan PIoU ---
# Ultralytics 8.4.x: TIDAK ada self.iou_loss/self.use_dfl/self._df_loss/self.reg_max.
# Yang ada: self.dfl_loss (DFLoss instance atau None).
# IoU dihitung via fungsi bbox_iou dari ultralytics.utils.metrics.
from ultralytics.utils.metrics import bbox_iou
from ultralytics.utils.tal import bbox2dist

_original_bbox_loss = ult_loss.BboxLoss.forward

def _piou_bbox_loss_forward(self, pred_dist, pred_bboxes, anchor_points,
                             target_bboxes, target_scores, target_scores_sum, fg_mask,
                             imgsz=None, stride=None):
    """Override BboxLoss.forward dengan PIoU. Kompatibel Ultralytics 8.4.x."""
    weight = target_scores.sum(-1)[fg_mask].unsqueeze(-1)
    # CIoU base (tetap dipakai sebagai komponen blending)
    iou = bbox_iou(pred_bboxes[fg_mask], target_bboxes[fg_mask], xywh=False, CIoU=True)

    # PIoU blended: 70% PIoU + 30% CIoU untuk stabilitas training
    piou_val = piou_loss(pred_bboxes[fg_mask], target_bboxes[fg_mask])
    loss_iou = ((1.0 - iou) * 0.3 + piou_val * 0.7) * weight
    loss_iou = loss_iou.sum() / target_scores_sum

    # DFL loss — gunakan self.dfl_loss (DFLoss instance) atau fallback L1
    if self.dfl_loss is not None:
        target_ltrb = bbox2dist(anchor_points, target_bboxes, self.dfl_loss.reg_max - 1)
        loss_dfl = self.dfl_loss(
            pred_dist[fg_mask].view(-1, self.dfl_loss.reg_max),
            target_ltrb[fg_mask]
        ) * weight
        loss_dfl = loss_dfl.sum() / target_scores_sum
    else:
        if imgsz is not None and stride is not None:
            tl = bbox2dist(anchor_points, target_bboxes) * stride
            tl[..., 0::2] /= imgsz[1]
            tl[..., 1::2] /= imgsz[0]
            pd = pred_dist * stride
            pd[..., 0::2] /= imgsz[1]
            pd[..., 1::2] /= imgsz[0]
            loss_dfl = (F.l1_loss(pd[fg_mask], tl[fg_mask], reduction='none')
                        .mean(-1, keepdim=True) * weight)
            loss_dfl = loss_dfl.sum() / target_scores_sum
        else:
            loss_dfl = torch.tensor(0., device=pred_dist.device)
    return loss_iou, loss_dfl

ult_loss.BboxLoss.forward = _piou_bbox_loss_forward
print('✓ PIoU berhasil di-patch ke BboxLoss (Ultralytics 8.4.x kompatibel)')


# --- Patch: tambahkan EMA setelah setiap C2f di neck model ---
class C2fWithEMA(nn.Module):
    """C2f + EMA attention wrapper untuk neck YOLOv8."""

    def __init__(self, c2f_module: C2f):
        super().__init__()
        self.c2f = c2f_module
        ch = c2f_module.cv2.conv.out_channels  # output channels C2f
        # Pastikan ch bisa dibagi 8; jika tidak, turunkan factor
        factor = 8
        while ch % factor != 0 and factor > 1:
            factor //= 2
        self.ema = EMA(channels=ch, factor=factor)

    def forward(self, x):
        return self.ema(self.c2f(x))


def inject_ema_to_neck(model):
    """
    Traverse model.model dan ganti semua C2f pada indeks neck (>10)
    dengan C2fWithEMA.
    """
    neck_start = 10   # pada YOLOv8n, layer 0-9 = backbone, >=10 = neck+head
    injected = 0
    for i, layer in enumerate(model.model.model):
        if i >= neck_start and isinstance(layer, C2f):
            model.model.model[i] = C2fWithEMA(layer)
            injected += 1
    print(f'✓ EMA disematkan ke {injected} C2f layer di neck')
    return model


print('\n✓ Modul EMA & PIoU siap. Gunakan inject_ema_to_neck(model) setelah load model.')

✓ PIoU berhasil di-patch ke BboxLoss (Ultralytics 8.4.x kompatibel)

✓ Modul EMA & PIoU siap. Gunakan inject_ema_to_neck(model) setelah load model.


## Cell 4 — Hyperparameter Tuning dengan Optuna

Mencari kombinasi terbaik:
- `lr0`, `weight_decay`, `batch`
- Loss coefficients: `box`, `cls`, `dfl`

Setiap trial melatih model selama 15 epoch (quick trial) lalu dievaluasi via `mAP50`.
Jalankan **satu kali**; hasilnya disimpan di `best_params`.

In [4]:
# ─── Recovery: ambil hasil trial dari folder (args.yaml + validasi best.pt) ─────
# Jalankan cell ini SETELAH Cell 2 & 3. Hasil: best_params.json + variabel best_params.
from pathlib import Path
import yaml
import json
from datetime import datetime
from ultralytics import YOLO

TUNE_DIR = OUTPUT_DIR / 'optuna_trials'
BEST_PARAMS_JSON = TUNE_DIR / 'best_params.json'
PARAM_KEYS = ['lr0', 'lrf', 'weight_decay', 'momentum', 'batch', 'box', 'cls', 'dfl']

def get_trial_params(args_path: Path):
    """Ambil hyperparameter Optuna dari args.yaml yang disimpan YOLO."""
    if not args_path.exists():
        return None
    with open(args_path) as f:
        args = yaml.safe_load(f)
    if not args:
        return None
    params = {}
    for k in PARAM_KEYS:
        if k in args:
            v = args[k]
            params[k] = int(round(v)) if k == 'batch' else float(v)
    return params if len(params) == len(PARAM_KEYS) else None

def get_map50_95_from_best_pt(weights_path: Path, data_yaml: str, device) -> float:
    """Validasi best.pt dan return mAP50-95."""
    model = YOLO(str(weights_path))
    metrics = model.val(data=data_yaml, batch=16, device=device, verbose=False)
    return float(metrics.results_dict.get('metrics/mAP50-95(B)', 0.0))

# Scan trial_000, trial_001, ...
trial_dirs = sorted([d for d in TUNE_DIR.iterdir() if d.is_dir() and d.name.startswith('trial_')])
print(f'Ditemukan {len(trial_dirs)} folder trial di {TUNE_DIR}')

results = []  # (trial_name, params, mAP50-95)
for td in trial_dirs:
    args_path = td / 'args.yaml'
    best_pt = td / 'weights' / 'best.pt'
    params = get_trial_params(args_path)
    if params is None:
        print(f'  Skip {td.name}: tidak ada args.yaml atau param tidak lengkap')
        continue
    if not best_pt.exists():
        print(f'  Skip {td.name}: best.pt belum ada (trial belum selesai?)')
        continue
    try:
        m = get_map50_95_from_best_pt(best_pt, str(DATA_YAML), DEFAULT_PARAMS['device'])
        results.append((td.name, params, m))
        print(f'  {td.name}: mAP50-95 = {m:.4f}')
    except Exception as e:
        print(f'  Skip {td.name}: {e}')

if not results:
    print('Tidak ada trial yang bisa dipakai. Pastikan ada args.yaml dan weights/best.pt per trial.')
    best_params = {}
else:
    # Trial dengan mAP50-95 tertinggi = best_params
    best_name, best_params, best_value = max(results, key=lambda x: x[2])
    with open(BEST_PARAMS_JSON, 'w') as f:
        json.dump(best_params, f, indent=2)
    print(f'\n═══ Best dari {len(results)} trial (recovery) ═══')
    print(f'  Trial terbaik: {best_name}  →  mAP50-95 = {best_value:.4f}')
    for k, v in best_params.items():
        print(f'  {k:20s}: {v}')
    print(f'\n✓ best_params tersimpan di {BEST_PARAMS_JSON}')
    # Opsional: isi Optuna study agar trial lama terhitung; lalu di Cell 4 set N_TRIALS=20 → hanya sisa yang jalan
    OPTUNA_DB = TUNE_DIR / 'optuna_study.db'
    if not OPTUNA_DB.exists() and results:
        import optuna
        from optuna.trial import create_trial
        from optuna.distributions import FloatDistribution, CategoricalDistribution
        storage = f'sqlite:///{OPTUNA_DB.as_posix()}'
        study = optuna.create_study(direction='maximize', study_name='yolov8n_acne_tuning',
                                    storage=storage, load_if_exists=False)
        dists = {
            'lr0': FloatDistribution(1e-4, 1e-1, log=True),
            'lrf': FloatDistribution(0.001, 0.1, log=True),
            'weight_decay': FloatDistribution(1e-5, 1e-2, log=True),
            'momentum': FloatDistribution(0.85, 0.98),
            'batch': CategoricalDistribution([8, 16, 32]),
            'box': FloatDistribution(4.0, 12.0),
            'cls': FloatDistribution(0.2, 4.0),
            'dfl': FloatDistribution(0.5, 3.0),
        }
        for tname, p, val in results:
            trial = create_trial(params=p, distributions=dists, value=val)
            study.add_trial(trial)
        print(f'  Study Optuna diisi {len(results)} trial → {OPTUNA_DB}. Jalankan Cell 4 dengan N_TRIALS=20 untuk tambah trial baru.')
    print('  Lanjut ke Cell 5 (training final) atau ubah N_TRIALS/TUNE_EPOCHS lalu jalankan Cell 4.')

Ditemukan 65 folder trial di /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/runs/optuna_trials
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 479.8±194.6 MB/s, size: 73.8 KB)
val: Scanning /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/test/labels.cache... 2328 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2328/2328 887.7Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 146/146 11.4it/s 12.8s0.2s
                   all       2328      26351      0.606      0.442      0.446      0.223
Speed: 0.5ms preprocess, 2.5ms inference, 0.0ms loss, 1.0ms postprocess per image
Results saved to /home/rof/Delima/Delima/runs/detect/val17
  trial_000: mAP50-95 = 0.2228
  Skip trial_001: best.pt belum ada (trial belum selesai?)
Ultralytics 8.4.22 🚀 Python-

In [4]:
import optuna
import wandb
from ultralytics import YOLO
from pathlib import Path
import shutil, os

# ─── Konfigurasi Optuna ──────────────────────────────────────────────────────
N_TRIALS      = 50       # Jumlah trial (kurangi jika GPU terbatas)
TUNE_EPOCHS   = 50      # Epoch cepat per trial
TUNE_IMG_SIZE = 640
TUNE_DEVICE   = DEFAULT_PARAMS['device']
TUNE_DIR      = OUTPUT_DIR / 'optuna_trials'
TUNE_DIR.mkdir(exist_ok=True)


def objective(trial: optuna.Trial) -> float:
    """Fungsi objektif Optuna — satu trial = satu YOLOv8n training singkat."""

    # --- Sample hyperparameters ---
    lr0          = trial.suggest_float('lr0',          1e-4,  1e-1,  log=True)
    lrf          = trial.suggest_float('lrf',          0.001, 0.1,   log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-5,  1e-2,  log=True)
    momentum     = trial.suggest_float('momentum',     0.85,  0.98)
    batch        = trial.suggest_categorical('batch',  [8, 16, 32])
    box          = trial.suggest_float('box',          4.0,   12.0)
    cls          = trial.suggest_float('cls',          0.2,   4.0)
    dfl          = trial.suggest_float('dfl',          0.5,   3.0)

    trial_name = f'trial_{trial.number:03d}'
    trial_dir  = TUNE_DIR / trial_name

    # Log ke WandB (satu run per trial)
    run = wandb.init(
        project = WANDB_PROJECT,
        entity  = WANDB_ENTITY,
        name    = trial_name,
        group   = 'optuna_tuning',
        config  = {
            'lr0': lr0, 'lrf': lrf, 'weight_decay': weight_decay,
            'momentum': momentum, 'batch': batch,
            'box': box, 'cls': cls, 'dfl': dfl,
            'epochs': TUNE_EPOCHS, 'imgsz': TUNE_IMG_SIZE,
        },
        reinit  = True,
    )

    try:
        model = YOLO('yolov8n.pt')          # transfer learning dari pretrained
        inject_ema_to_neck(model)            # tambahkan EMA di neck

        results = model.train(
            data         = str(DATA_YAML),
            epochs       = TUNE_EPOCHS,
            imgsz        = TUNE_IMG_SIZE,
            batch        = batch,
            lr0          = lr0,
            lrf          = lrf,
            momentum     = momentum,
            weight_decay = weight_decay,
            box          = box,
            cls          = cls,
            dfl          = dfl,
            device       = TUNE_DEVICE,
            project      = str(TUNE_DIR),
            name         = trial_name,
            exist_ok     = True,
            verbose      = False,
            workers      = 4,
            patience     = 5,   # early stop cepat di trial
            save         = False,
            plots        = False,
        )

        # Ambil mAP50 dari hasil validasi terakhir
        val_metrics = results.results_dict
        map50 = float(val_metrics.get('metrics/mAP50(B)', 0.0))
        map50_95 = float(val_metrics.get('metrics/mAP50-95(B)', 0.0))

        wandb.log({'mAP50': map50, 'mAP50_95': map50_95,
                   'fitness': map50 * 0.1 + map50_95 * 0.9})
        run.finish()
        return map50_95   # Optuna memaksimalkan mAP50-95

    except Exception as e:
        print(f'  [Trial {trial.number}] Error: {e}')
        run.finish()
        return 0.0


# ─── Jalankan Optuna ────────────────────────────────────────────────────────
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(
    direction  = 'maximize',
    study_name = 'yolov8n_acne_tuning',
    sampler    = optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_params = study.best_params
print('\n═══ Best Hyperparameters ═══')
for k, v in best_params.items():
    print(f'  {k:20s}: {v}')
print(f'  Best mAP50-95 (trial)  : {study.best_value:.4f}')

/home/rof/.conda/envs/yolo_gpu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


✓ EMA disematkan ke 4 C2f layer di neck
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=10.92940916619948, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.4842370446241935, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/runs/data_runtime.yaml, degrees=0.0, deterministic=True, device=0, dfl=2.2701814444901136, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0013292918943162175, lrf=0.07969454818643935, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, 

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


fitness,▁
mAP50,▁
mAP50_95,▁
fitness,0.24518
mAP50,0.44601
mAP50_95,0.22286


Best trial: 0. Best value: 0.222865:   2%|▊                                       | 1/50 [1:11:29<58:23:12, 4289.64s/it]

...

Best trial: 13. Best value: 0.611399:  34%|████████████▌                        | 17/50 [17:39:17<34:47:57, 3796.28s/it]

✓ EMA disematkan ke 4 C2f layer di neck
New https://pypi.org/project/ultralytics/8.4.23 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.003649649139596, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.8497448642202542, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/runs/data_runtime.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.9601405672941843, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.004169371506

Best trial: 13. Best value: 0.611399:  34%|████████████▌                        | 17/50 [18:24:27<34:47:57, 3796.28s/it]Exception in thread Thread-93 (_pin_memory_loop):
Traceback (most recent call last):
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 52, in _pin_memory_loop
                                                                                                                            do_one_step()
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/site-packages/torch/utils/data/_utils/pin_memory.py", line 28, in do_one_step
r = in_queue.get(timeout=MP_STATUS_CHECK_INTERVAL)
  File "/home/rof/.conda/envs/yolo_gpu/lib/python3.10/multiprocessing/queues.py", line 122

[W 2026-03-17 10:10:00,923] Trial 17 failed with parameters: {'lr0': 0.004169371506212444, 'lrf': 0.012946404316262265, 'weight_decay': 0.0007726556738189388, 'momentum': 0.9358388920596771, 'batch': 16, 'box': 7.003649649139596, 'cls': 0.8497448642202542, 'dfl': 1.9601405672941843} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/rof/.local/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_156460/4086787370.py", line 51, in objective
    results = model.train(
  File "/home/rof/.local/lib/python3.10/site-packages/ultralytics/engine/model.py", line 777, in train
    self.trainer.train()
  File "/home/rof/.local/lib/python3.10/site-packages/ultralytics/engine/trainer.py", line 244, in train
    self._do_train()
  File "/home/rof/.local/lib/python3.10/site-packages/ultralytics/engine/trainer.py", line 481, in _do_train
    pbar.set_description(
  Fi

KeyboardInterrupt: 

## Cell 5 — Training Final dengan Best Params + WandB Logging Lengkap

In [5]:
from ultralytics import YOLO
from ultralytics import settings as ult_settings
from pathlib import Path
import torch, os

# ─── Matikan SEMUA integrasi WandB Ultralytics ──────────────────────────────
ult_settings.update({'wandb': False})
os.environ['WANDB_DISABLED'] = 'true'   # pastikan env var juga mati

# ─── Merge default + best_params dari Optuna / Recovery (Cell 3.5 atau Cell 4) ─
# Jika best_params belum ada (kernel baru / belum jalankan tuning), load dari file atau study.
try:
    _ = best_params
except NameError:
    best_params = {}
    _tune_dir = OUTPUT_DIR / 'optuna_trials'
    _json_path = _tune_dir / 'best_params.json'
    if _json_path.exists():
        import json
        with open(_json_path) as f:
            best_params = json.load(f)
        print(f'✓ best_params dimuat dari {_json_path} (dari Cell 3.5 recovery atau Cell 4)')
    else:
        try:
            import optuna
            _db = _tune_dir / 'optuna_study.db'
            if _db.exists():
                _storage = f'sqlite:///{_db.as_posix()}'
                study = optuna.load_study(study_name='yolov8n_acne_tuning', storage=_storage)
                if len(study.trials) > 0:
                    best_params = study.best_params
                    print(f'✓ best_params dimuat dari study Optuna ({len(study.trials)} trial)')
        except Exception as e:
            print(f'Tidak bisa load study: {e}')
    if not best_params:
        print('⚠ best_params kosong; menggunakan DEFAULT_PARAMS saja.')

TRAIN_PARAMS = {**DEFAULT_PARAMS, **best_params}

# ─── Augmentasi ──────────────────────────────────────────────────────────────
AUG_PARAMS = dict(
    hsv_h      = 0.015,
    hsv_s      = 0.7,
    hsv_v      = 0.4,
    degrees    = 10.0,
    translate  = 0.1,
    scale      = 0.5,
    flipud     = 0.0,
    fliplr     = 0.5,
    mosaic     = 1.0,
    mixup      = 0.1,
    copy_paste = 0.1,
)

FINAL_PARAMS = {**TRAIN_PARAMS, **AUG_PARAMS}

print('─── Parameter Training Final ───')
for k, v in FINAL_PARAMS.items():
    print(f'  {k:20s}: {v}')

# ─── Load model & inject EMA ────────────────────────────────────────────────
model = YOLO('yolov8n.pt')
inject_ema_to_neck(model)

# ─── Training (tanpa WandB aktif agar tidak bentrok) ────────────────────────
print('\n🚀 Memulai training...')
results = model.train(
    data         = str(DATA_YAML),
    epochs       = FINAL_PARAMS['epochs'],
    imgsz        = FINAL_PARAMS['imgsz'],
    batch        = FINAL_PARAMS['batch'],
    lr0          = FINAL_PARAMS['lr0'],
    lrf          = FINAL_PARAMS['lrf'],
    momentum     = FINAL_PARAMS['momentum'],
    weight_decay = FINAL_PARAMS['weight_decay'],
    box          = FINAL_PARAMS['box'],
    cls          = FINAL_PARAMS['cls'],
    dfl          = FINAL_PARAMS['dfl'],
    device       = FINAL_PARAMS['device'],
    workers      = FINAL_PARAMS['workers'],
    patience     = FINAL_PARAMS['patience'],
    hsv_h        = AUG_PARAMS['hsv_h'],
    hsv_s        = AUG_PARAMS['hsv_s'],
    hsv_v        = AUG_PARAMS['hsv_v'],
    degrees      = AUG_PARAMS['degrees'],
    translate    = AUG_PARAMS['translate'],
    scale        = AUG_PARAMS['scale'],
    flipud       = AUG_PARAMS['flipud'],
    fliplr       = AUG_PARAMS['fliplr'],
    mosaic       = AUG_PARAMS['mosaic'],
    mixup        = AUG_PARAMS['mixup'],
    copy_paste   = AUG_PARAMS['copy_paste'],
    project      = str(OUTPUT_DIR),
    name         = 'final_ema_piou',
    exist_ok     = True,
    save         = True,
    plots        = True,
    verbose      = True,
)

# ─── Simpan path model terbaik ──────────────────────────────────────────────
BEST_MODEL_PATH = Path(results.save_dir) / 'weights' / 'best.pt'
TRAIN_SAVE_DIR  = Path(results.save_dir)
print(f'\n✓ Training selesai. Best model: {BEST_MODEL_PATH}')

# ─── Re-aktifkan WandB dan log semua hasil (run baru, jadi tidak bentrok) ───
os.environ.pop('WANDB_DISABLED', None)
import wandb

train_run = wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = 'final_training_ema_piou',
    group   = 'final',
    config  = {
        **FINAL_PARAMS,
        'model'      : 'yolov8n',
        'neck_module': 'EMA',
        'iou_loss'   : 'PIoU',
        'num_classes': NUM_CLASSES,
        'class_names': CLASS_NAMES,
        'dataset'    : str(DATA_YAML),
    },
)
print('WandB run URL:', train_run.url)

# Log metrics akhir
final_rd = results.results_dict if hasattr(results, 'results_dict') else {}
train_run.log({
    'final/precision' : final_rd.get('metrics/precision(B)', 0),
    'final/recall'    : final_rd.get('metrics/recall(B)', 0),
    'final/mAP50'     : final_rd.get('metrics/mAP50(B)', 0),
    'final/mAP50_95'  : final_rd.get('metrics/mAP50-95(B)', 0),
    'final/box_loss'  : final_rd.get('val/box_loss', 0),
    'final/cls_loss'  : final_rd.get('val/cls_loss', 0),
    'final/dfl_loss'  : final_rd.get('val/dfl_loss', 0),
})

# Log semua plot hasil training
for _img_path in TRAIN_SAVE_DIR.glob('*.png'):
    train_run.log({_img_path.stem: wandb.Image(str(_img_path))})
    print(f'  ✓ Plot: {_img_path.name}')

# Log artifact model
if BEST_MODEL_PATH.exists():
    artifact = wandb.Artifact(
        name        = 'yolov8n_ema_piou_best',
        type        = 'model',
        description = 'YOLOv8n + EMA neck + PIoU loss — best weights',
        metadata    = {**FINAL_PARAMS, 'class_names': CLASS_NAMES},
    )
    artifact.add_file(str(BEST_MODEL_PATH))
    train_run.log_artifact(artifact)
    print('✓ Model artifact diunggah ke WandB')

train_run.finish()
print('✓ WandB run training selesai')

─── Parameter Training Final ───
  epochs              : 200
  imgsz               : 640
  device              : 0
  workers             : 4
  patience            : 20
  lr0                 : 0.004942536833127779
  lrf                 : 0.0011672711987380454
  momentum            : 0.9425571142877435
  weight_decay        : 0.0006566834816537323
  batch               : 16
  box                 : 6.028358163909732
  cls                 : 0.7500457806104692
  dfl                 : 2.180837687691078
  hsv_h               : 0.015
  hsv_s               : 0.7
  hsv_v               : 0.4
  degrees             : 10.0
  translate           : 0.1
  scale               : 0.5
  flipud              : 0.0
  fliplr              : 0.5
  mosaic              : 1.0
  mixup               : 0.1
  copy_paste          : 0.1
✓ EMA disematkan ke 4 C2f layer di neck

🚀 Memulai training...
New https://pypi.org/project/ultralytics/8.4.23 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.22 🚀 Py

WandB run URL: https://wandb.ai/delimabrpurba72-universitas-prasetiya-mulya/acne-detection-yolov8n/runs/xprojdka
  ✓ Plot: confusion_matrix_normalized.png
  ✓ Plot: results.png
  ✓ Plot: BoxP_curve.png
  ✓ Plot: BoxR_curve.png
  ✓ Plot: BoxF1_curve.png
  ✓ Plot: confusion_matrix.png
  ✓ Plot: BoxPR_curve.png


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


✓ Model artifact diunggah ke WandB


final/box_loss,▁
final/cls_loss,▁
final/dfl_loss,▁
final/mAP50,▁
final/mAP50_95,▁
final/precision,▁
final/recall,▁
final/box_loss,0
final/cls_loss,0
final/dfl_loss,0
final/mAP50,0.85711


✓ WandB run training selesai


## Cell 6 — Evaluasi pada Data Train & Data Test + Visualisasi

Evaluasi pada **data train** (untuk melihat fit) dan **data test** (unseen, untuk generalisasi). Keduanya dilaporkan dan di-log ke WandB.

In [6]:
import wandb
import yaml
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from pathlib import Path
from ultralytics import YOLO
import torch

# ─── Load model terbaik ─────────────────────────────────────────────────────
# Gunakan BEST_MODEL_PATH dari Cell 5, atau ubah path manual di bawah
try:
    _best_path = BEST_MODEL_PATH
except NameError:
    # Fallback: cari otomatis
    candidates = list((OUTPUT_DIR / 'final_ema_piou' / 'weights').glob('best.pt'))
    if not candidates:
        candidates = list(OUTPUT_DIR.rglob('best.pt'))
    assert candidates, 'best.pt tidak ditemukan! Jalankan Cell 5 terlebih dahulu.'
    _best_path = candidates[-1]

print(f'Evaluasi model: {_best_path}')
eval_model = YOLO(str(_best_path))

# ─── Buat data.yaml untuk evaluasi (train & test) ──────────────────────────────
with open(DATA_YAML) as f:
    base_cfg = yaml.safe_load(f)

test_cfg = base_cfg.copy()
test_cfg['val'] = str(BASE_DIR / 'test' / 'images')
TEST_YAML = OUTPUT_DIR / 'test_eval.yaml'
with open(TEST_YAML, 'w') as f:
    yaml.dump(test_cfg, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f'Test yaml: {TEST_YAML}')

train_cfg = base_cfg.copy()
train_cfg['val'] = str(BASE_DIR / 'train' / 'images')
TRAIN_YAML = OUTPUT_DIR / 'train_eval.yaml'
with open(TRAIN_YAML, 'w') as f:
    yaml.dump(train_cfg, f, default_flow_style=False, allow_unicode=True, sort_keys=False)
print(f'Train yaml: {TRAIN_YAML}')

# ─── WandB run evaluasi ─────────────────────────────────────────────────────
eval_run = wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = 'evaluation_train_and_test',
    group   = 'evaluation',
    reinit  = True,
)

# ─── Evaluasi pada DATA TRAIN ───────────────────────────────────────────────
print('\n─── Evaluasi pada Data TRAIN ───')
metrics_train = eval_model.val(
    data     = str(TRAIN_YAML),
    imgsz    = DEFAULT_PARAMS['imgsz'],
    batch    = 16,
    device   = DEFAULT_PARAMS['device'],
    split    = 'val',
    save_json= True,
    plots    = True,
    verbose  = True,
    project  = str(OUTPUT_DIR),
    name     = 'eval_train',
    exist_ok = True,
)
rd_train   = metrics_train.results_dict
prec_train = rd_train.get('metrics/precision(B)', 0)
rec_train  = rd_train.get('metrics/recall(B)', 0)
f1_train   = 2 * prec_train * rec_train / (prec_train + rec_train + 1e-9)

# ─── Evaluasi pada DATA TEST (unseen) ───────────────────────────────────────
print('\n─── Evaluasi pada Data TEST (unseen) ───')
metrics = eval_model.val(
    data     = str(TEST_YAML),
    imgsz    = DEFAULT_PARAMS['imgsz'],
    batch    = 16,
    device   = DEFAULT_PARAMS['device'],
    split    = 'val',
    save_json= True,
    plots    = True,
    verbose  = True,
    project  = str(OUTPUT_DIR),
    name     = 'eval_test',
    exist_ok = True,
)
rd   = metrics.results_dict
prec = rd.get('metrics/precision(B)', 0)
rec  = rd.get('metrics/recall(B)', 0)
f1   = 2 * prec * rec / (prec + rec + 1e-9)

# ─── Helper: ekstrak IoU per kelas dari results ─────────────────────────────
def get_per_class_iou(val_metrics, class_names):
    """
    Hitung IoU per kelas dari hasil val().
    IoU per kelas ≈ rata-rata IoU dari setiap matched prediction.
    Menggunakan ap_class_index dan maps (mAP per kelas) sebagai proxy IoU.
    """
    iou_dict = {}
    try:
        # maps: mAP50-95 per kelas, tersedia di results object
        maps = val_metrics.maps          # array panjang nc, mAP50-95 per kelas
        ap50 = val_metrics.ap50()        # mAP50 per kelas
        box  = val_metrics.box           # DetMetrics box object
        p_cls  = box.p                   # precision per kelas
        r_cls  = box.r                   # recall per kelas
        ap_cls = box.ap                  # AP50-95 per kelas
        ap50_cls = box.ap50()            # AP50 per kelas
        for i, name in enumerate(class_names):
            if i < len(maps):
                p_val  = float(p_cls[i])  if i < len(p_cls)  else 0.0
                r_val  = float(r_cls[i])  if i < len(r_cls)  else 0.0
                f1_val = 2 * p_val * r_val / (p_val + r_val + 1e-9)
                iou_dict[name] = {
                    'Precision': p_val,
                    'Recall'   : r_val,
                    'F1'       : f1_val,
                    'mAP@0.5'  : float(ap50_cls[i]) if i < len(ap50_cls) else 0.0,
                    'mAP@0.5:0.95': float(maps[i]),
                }
    except Exception as e:
        print(f'  (IoU per kelas tidak dapat diekstrak: {e})')
    return iou_dict

per_class_train = get_per_class_iou(metrics_train, CLASS_NAMES)
per_class_test  = get_per_class_iou(metrics,       CLASS_NAMES)

# ─── Tabel perbandingan Train vs Test (overall) ─────────────────────────────
W = 60
print('\n' + '═'*W)
print('  PERBANDINGAN EVALUASI — DATA TRAIN vs DATA TEST')
print('═'*W)
print(f'  {"Metrik":<22} {"Train":>12} {"Test":>12}')
print('─'*W)
print(f'  {"Precision":<22} {prec_train:>12.4f} {prec:>12.4f}')
print(f'  {"Recall":<22} {rec_train:>12.4f} {rec:>12.4f}')
print(f'  {"F1-Score":<22} {f1_train:>12.4f} {f1:>12.4f}')
print(f'  {"mAP@0.5":<22} {rd_train.get("metrics/mAP50(B)", 0):>12.4f} {rd.get("metrics/mAP50(B)", 0):>12.4f}')
print(f'  {"mAP@0.5:0.95":<22} {rd_train.get("metrics/mAP50-95(B)", 0):>12.4f} {rd.get("metrics/mAP50-95(B)", 0):>12.4f}')
print('═'*W)

# ─── Tabel IoU / Metrik per kelas (TEST) ─────────────────────────────────────
if per_class_test:
    print('\n' + '═'*80)
    print('  METRIK PER KELAS — DATA TEST')
    print('═'*80)
    print(f'  {"Kelas":<25} {"Precision":>10} {"Recall":>10} {"F1":>10} {"mAP@0.5":>10} {"mAP@0.5:95":>12}')
    print('─'*80)
    for cls_name, m in per_class_test.items():
        print(f'  {cls_name:<25} {m["Precision"]:>10.4f} {m["Recall"]:>10.4f} '
              f'{m["F1"]:>10.4f} {m["mAP@0.5"]:>10.4f} {m["mAP@0.5:0.95"]:>12.4f}')
    print('═'*80)

# ─── Tabel IoU / Metrik per kelas (TRAIN) ────────────────────────────────────
if per_class_train:
    print('\n' + '═'*80)
    print('  METRIK PER KELAS — DATA TRAIN')
    print('═'*80)
    print(f'  {"Kelas":<25} {"Precision":>10} {"Recall":>10} {"F1":>10} {"mAP@0.5":>10} {"mAP@0.5:95":>12}')
    print('─'*80)
    for cls_name, m in per_class_train.items():
        print(f'  {cls_name:<25} {m["Precision"]:>10.4f} {m["Recall"]:>10.4f} '
              f'{m["F1"]:>10.4f} {m["mAP@0.5"]:>10.4f} {m["mAP@0.5:0.95"]:>12.4f}')
    print('═'*80)

# ─── Log ke WandB ───────────────────────────────────────────────────────────
_wandb_log = {
    'eval/train_precision' : prec_train,
    'eval/train_recall'    : rec_train,
    'eval/train_f1'        : f1_train,
    'eval/train_mAP50'     : rd_train.get('metrics/mAP50(B)', 0),
    'eval/train_mAP50_95'  : rd_train.get('metrics/mAP50-95(B)', 0),
    'eval/test_precision'  : prec,
    'eval/test_recall'     : rec,
    'eval/test_f1'         : f1,
    'eval/test_mAP50'      : rd.get('metrics/mAP50(B)', 0),
    'eval/test_mAP50_95'   : rd.get('metrics/mAP50-95(B)', 0),
}
# Per-kelas test
for cls_name, m in per_class_test.items():
    for metric_name, val in m.items():
        _wandb_log[f'per_class_test/{cls_name}/{metric_name}'] = val
# Per-kelas train
for cls_name, m in per_class_train.items():
    for metric_name, val in m.items():
        _wandb_log[f'per_class_train/{cls_name}/{metric_name}'] = val

eval_run.log(_wandb_log)

# ─── Model size & GFLOPs ────────────────────────────────────────────────────
n_params = sum(p.numel() for p in eval_model.model.parameters()) / 1e6
print(f'\n  Parameter count : {n_params:.2f} M')

gflops = 0.0
try:
    # Ultralytics 8.4.x: info() mengembalikan (params, gflops)
    _info = eval_model.info(verbose=False)
    if isinstance(_info, (list, tuple)) and len(_info) >= 2:
        gflops = float(_info[1])
    if gflops == 0.0:
        from ultralytics.utils.torch_utils import get_flops
        gflops = get_flops(eval_model.model, imgsz=640)
    print(f'  GFLOPs          : {gflops:.2f}')
except Exception as _e:
    print(f'  GFLOPs          : tidak dapat dihitung ({_e})')

eval_run.log({'model/params_M': n_params, 'model/GFLOPs': gflops})

# ─── Inference time ─────────────────────────────────────────────────────────
import time
test_img_dir = BASE_DIR / 'test' / 'images'
sample_imgs  = list(test_img_dir.glob('*.jpg'))[:5] + list(test_img_dir.glob('*.png'))[:5]
sample_imgs  = sample_imgs[:5]

if sample_imgs:
    t0 = time.perf_counter()
    _ = eval_model.predict(source=[str(p) for p in sample_imgs],
                            imgsz=640, verbose=False, device=DEFAULT_PARAMS['device'])
    elapsed = (time.perf_counter() - t0) / len(sample_imgs) * 1000
    print(f'  Inference latency: {elapsed:.1f} ms/gambar')
    eval_run.log({'model/inference_ms': elapsed})

# ─── Upload grafik evaluasi ke WandB ────────────────────────────────────────
save_dir = Path(metrics.save_dir)           # eval_test
save_dir_train = Path(metrics_train.save_dir)  # eval_train
plots_to_upload = [
    ('confusion_matrix.png',    'Confusion Matrix (Test)'),
    ('confusion_matrix_normalized.png', 'Confusion Matrix Normalized (Test)'),
    ('PR_curve.png',            'PR Curve (Test)'),
    ('F1_curve.png',            'F1–Confidence Curve (Test)'),
    ('P_curve.png',             'Precision–Confidence (Test)'),
    ('R_curve.png',             'Recall–Confidence (Test)'),
    ('results.png',             'Training & Validation Loss Curves'),
]

for fname, caption in plots_to_upload:
    fpath = save_dir / fname
    if not fpath.exists():
        try:
            fpath = list((OUTPUT_DIR / 'final_ema_piou').glob(fname))[0]
        except Exception:
            continue
    if fpath.exists():
        eval_run.log({caption: wandb.Image(str(fpath), caption=caption)})

# Upload grafik evaluasi TRAIN (confusion matrix)
for fname, caption in [('confusion_matrix.png', 'Confusion Matrix (Train)'),
                       ('confusion_matrix_normalized.png', 'Confusion Matrix Normalized (Train)')]:
    fpath = save_dir_train / fname
    if fpath.exists():
        eval_run.log({caption: wandb.Image(str(fpath), caption=caption)})

print('\n✓ Semua grafik evaluasi (train & test) diunggah ke WandB')

# ─── Tampilkan grafik lokal ──────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

display_plots = [save_dir / f for f, _c in plots_to_upload if (save_dir / f).exists()]
for i, img_path in enumerate(display_plots[:6]):
    img = mpimg.imread(str(img_path))
    axes[i].imshow(img)
    axes[i].set_title(img_path.stem.replace('_', ' '), fontsize=11)
    axes[i].axis('off')
for j in range(len(display_plots[:6]), 6):
    axes[j].axis('off')

plt.suptitle('Hasil Evaluasi YOLOv8n + EMA + PIoU', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'evaluation_summary.png'), dpi=150, bbox_inches='tight')
plt.show()

eval_run.finish()
print('\n✓ Evaluasi selesai. Semua hasil tersimpan di WandB.')

Evaluasi model: /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/runs/final_ema_piou/weights/best.pt
Test yaml: /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/runs/test_eval.yaml
Train yaml: /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/runs/train_eval.yaml


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.



─── Evaluasi pada Data TRAIN ───
Ultralytics 8.4.22 🚀 Python-3.10.19 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3060, 11908MiB)
Model summary (fused): 73 layers, 3,007,403 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3680.4±1561.7 MB/s, size: 49.6 KB)
val: Scanning /home/rof/Delima/Delima/DATA FIX UNTUK TRAIN/train/labels.cache... 9312 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 9312/9312 2.4Git/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 582/582 11.2it/s 52.2s<0.2s
                   all       9312     104434      0.915      0.837      0.915      0.675
         atrophic_scar       3236      21974      0.853      0.734      0.854       0.55
                comedo       3556      24090      0.913      0.662      0.837       0.56
     hypertrophic_scar       1105      10006      0.949      0.952      0.981      0.774
               melasma       1223       9579     

<Figure size 1800x1000 with 6 Axes>

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/test_f1,▁
eval/test_mAP50,▁
eval/test_mAP50_95,▁
eval/test_precision,▁
eval/test_recall,▁
eval/train_f1,▁
eval/train_mAP50,▁
eval/train_mAP50_95,▁
eval/train_precision,▁
eval/train_recall,▁
+3,...



✓ Evaluasi selesai. Semua hasil tersimpan di WandB.


## Cell 7 — Inference Visual (Ground Truth vs Prediksi)

Visualisasi beberapa gambar test dengan bounding box prediksi untuk analisis kualitatif.

In [7]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.lines import Line2D
from pathlib import Path
from PIL import Image
import numpy as np
import random, wandb, math
from ultralytics import YOLO

try:
    _best_path = BEST_MODEL_PATH
except NameError:
    candidates = list((OUTPUT_DIR / 'final_ema_piou' / 'weights').glob('best.pt'))
    if not candidates:
        candidates = list(OUTPUT_DIR.rglob('best.pt'))
    assert candidates, 'best.pt tidak ditemukan! Jalankan Cell 5 (training final) dulu.'
    _best_path = candidates[-1]

inf_model = YOLO(str(_best_path))

# ─── Ambil sampel gambar test ────────────────────────────────────────────────
test_img_dir  = BASE_DIR / 'test' / 'images'
test_lbl_dir  = BASE_DIR / 'test' / 'labels'
all_imgs      = list(test_img_dir.glob('*.jpg')) + list(test_img_dir.glob('*.png'))
random.seed(42)
sample_paths  = random.sample(all_imgs, min(8, len(all_imgs)))

COLORS = plt.cm.get_cmap('tab10', NUM_CLASSES)

# ─── Inference ───────────────────────────────────────────────────────────────
preds = inf_model.predict(
    source  = [str(p) for p in sample_paths],
    imgsz   = 640,
    conf    = 0.25,
    iou     = 0.45,
    device  = DEFAULT_PARAMS['device'],
    verbose = False,
)

# ─── Plot ────────────────────────────────────────────────────────────────────
n_cols = 4
n_rows = math.ceil(len(sample_paths) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
axes = np.array(axes).flatten()

wandb_images = []
for idx, (img_path, result) in enumerate(zip(sample_paths, preds)):
    ax = axes[idx]
    img_arr = np.array(Image.open(img_path).convert('RGB'))
    ih, iw  = img_arr.shape[:2]
    ax.imshow(img_arr)

    # Ground truth
    lbl_path = test_lbl_dir / (img_path.stem + '.txt')
    if lbl_path.exists():
        with open(lbl_path) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:5])
                x1 = (xc - bw/2) * iw; y1 = (yc - bh/2) * ih
                w  = bw * iw;           h  = bh * ih
                rect = patches.Rectangle((x1, y1), w, h,
                    linewidth=1.5, edgecolor='lime', facecolor='none',
                    linestyle='--')
                ax.add_patch(rect)

    # Prediksi
    if result.boxes is not None and len(result.boxes) > 0:
        boxes_xyxy = result.boxes.xyxy.cpu().numpy()
        confs      = result.boxes.conf.cpu().numpy()
        cls_ids    = result.boxes.cls.cpu().numpy().astype(int)
        for (x1, y1, x2, y2), conf, cid in zip(boxes_xyxy, confs, cls_ids):
            color = COLORS(cid)
            rect  = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor=color, facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1 - 4,
                    f'{CLASS_NAMES[cid]} {conf:.2f}',
                    fontsize=7, color='white',
                    bbox=dict(facecolor=color, alpha=0.7, pad=1, edgecolor='none'))

    ax.set_title(img_path.stem[:30], fontsize=8)
    ax.axis('off')

    # Simpan untuk WandB
    wandb_images.append(wandb.Image(img_arr,
        caption=f'{img_path.name} | pred boxes: {len(result.boxes) if result.boxes else 0}'))

# Legend
legend_lines = [
    Line2D([0], [0], color='lime',  linestyle='--', lw=2, label='Ground Truth'),
    Line2D([0], [0], color='white', lw=2, label='Prediksi'),
]
fig.legend(handles=legend_lines, loc='lower center', ncol=2, fontsize=10)

for j in range(len(sample_paths), len(axes)):
    axes[j].axis('off')

plt.suptitle('Inference Visual: Ground Truth (hijau putus) vs Prediksi (warna per kelas)',
             fontsize=12, fontweight='bold')
plt.tight_layout(rect=[0, 0.04, 1, 1])
plt.savefig(str(OUTPUT_DIR / 'inference_visual.png'), dpi=150, bbox_inches='tight')
plt.show()

# ─── Log ke WandB ────────────────────────────────────────────────────────────
vis_run = wandb.init(
    project  = WANDB_PROJECT,
    entity   = WANDB_ENTITY,
    name     = 'inference_visual',
    group    = 'evaluation',
    reinit   = True,
)
vis_run.log({'inference_samples': wandb_images})
vis_run.log({'inference_grid': wandb.Image(str(OUTPUT_DIR / 'inference_visual.png'))})
vis_run.finish()
print('\n✓ Inference visual selesai & diunggah ke WandB')

/tmp/ipykernel_197070/1615420754.py:28: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  COLORS = plt.cm.get_cmap('tab10', NUM_CLASSES)


<Figure size 2000x1000 with 8 Axes>

wandb: ERROR The nbformat package was not found. It is required to save notebook history.



✓ Inference visual selesai & diunggah ke WandB
